# Effective AI Agent Patterns using CrewAI

In [20]:
# --- SETUP ---
!pip install -q crewai openai wikipedia duckduckgo-search langchain-community

import os
from getpass import getpass

In [2]:
os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API Key: ")

🔑 Enter your OpenAI API Key: ··········


In [4]:
!pip install -q langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00


In [21]:
# --- Imports ---
from crewai import Agent, Task, Crew
from langchain_community.llms import OpenAI
from langchain_community.tools.duckduckgo_search import DuckDuckGoSearchResults
from langchain_community.utilities.wikipedia import WikipediaAPIWrapper
from langchain.agents.tools import Tool

ModuleNotFoundError: No module named 'langchain_community.tools.duckduckgo_search'

In [16]:
# --- TOOLS SETUP ---
duckduckgo_tool = Tool(
    name="DuckDuckGo",
    func=DuckDuckGoSearchRun().run,
    description="Search the internet using DuckDuckGo"
)

wikipedia_tool = Tool(
    name="Wikipedia",
    func=WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(lang="en")).run,
    description="Retrieve information from Wikipedia"
)

In [17]:
# --- Convert tools to CrewAI format ---
crewai_duckduckgo_tool = langchain_tool_to_crewai_tool(duckduckgo_tool)
crewai_wikipedia_tool = langchain_tool_to_crewai_tool(wikipedia_tool)

In [18]:
# --- LLM ---
llm = OpenAI(temperature=0)

In [19]:
# -----------------------------
# 1. ReAct-like Crew Task Setup
# -----------------------------
researcher = Agent(
    role="Researcher",
    goal="Use the internet and Wikipedia to gather accurate information",
    # Use the converted tools here:
    tools=[crewai_duckduckgo_tool, crewai_wikipedia_tool],
    backstory="Expert researcher in current affairs and history.",
    verbose=True,
    llm=llm,
)

analyst = Agent(
    role="Analyst",
    goal="Perform calculations and analyze results",
    tools=[],
    backstory="Great at mathematical reasoning.",
    verbose=True,
    llm=llm,
)

task1 = Task(
    description="Find the population of France and calculate its square root.",
    expected_output="Numeric output + reasoning",
    agent=researcher,
)

task2 = Task(
    description="Use population from Task 1 and divide it by 7. Summarize what this means.",
    expected_output="Final answer with interpretation",
    agent=analyst,
)

crew = Crew(
    agents=[researcher, analyst],
    tasks=[task1, task2],
    verbose=True,
)

results = crew.kickoff()
print(results)

TypeError: Can't instantiate abstract class BaseTool with abstract method _run

In [ ]:
# -----------------------------------
# 2. Multi-Agent Collaboration Example
# -----------------------------------
planner = Agent(
    role="Planner",
    goal="Break a complex query into subtasks",
    tools=[],
    backstory="Expert in strategy and workflow design.",
    verbose=True,
    llm=llm,
)

research_task = Task(
    description="Find GDP of Germany and Japan",
    expected_output="GDP values for each country",
    agent=researcher,
)

analysis_task = Task(
    description="Analyze and compare the GDPs",
    expected_output="Insightful comparison",
    agent=analyst,
)

crew_multi = Crew(
    agents=[planner, researcher, analyst],
    tasks=[research_task, analysis_task],
    verbose=True,
)

results_multi = crew_multi.kickoff()
print(results_multi)

In [ ]:
# -----------------------------
# 3. Plan-and-Execute via Crew
# -----------------------------
plan_task = Task(
    description="Given a goal: 'Compare inflation trends of USA and India', generate subtasks.",
    expected_output="List of subtasks",
    agent=planner,
)

crew_plan_exec = Crew(
    agents=[planner],
    tasks=[plan_task],
    verbose=True,
)

print(crew_plan_exec.kickoff())